Without AQE

In [0]:
#Disable AQE
spark.conf.set("spark.sql.adaptive.enabled","false")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled","false")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled","false")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold",-1)
spark.conf.set("spark.sql.optimizer.dynamicPartitionPruning.enabled","false")

In [0]:
#Enable AQE
spark.conf.set("spark.sql.adaptive.enabled","true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled","true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled","true")
#spark.conf.set("spark.sql.autoBroadcastJoinThreshold",10mb)
spark.conf.set("spark.sql.optimizer.dynamicPartitionPruning.enabled","true")

In [0]:
data_list = [
    (1, "A", 45),
    (2, "B", 55),
    (3, "C", 65),
    (4, "D", 18),
    (1, "A", 45),
    (2, "B", 55),
    (3, "C", 65),
    (4, "D", 18),
    (1, "A", 45),
    (2, "B", 55),
    (3, "C", 65),
    (4, "D", 18)
    ]
schema = "cid int, cname string, revenue int"
df = spark.createDataFrame(data_list, schema)

In [0]:
from pyspark.sql.functions import spark_partition_id,expr
df.select(expr("spark_partition_id() as partition_id")).groupBy("partition_id").count().show()

In [0]:
sum_df = df.groupBy("cname").sum("revenue")

In [0]:
sum_df.display()

In [0]:
sum_df.select(expr("spark_partition_id() as partition_id")).groupBy("partition_id").count().show()

In [0]:
spark.conf.set("spark.sql.shuffle.partitions","50")

In [0]:
df_shuff = spark.createDataFrame(data_list, schema)
sum_df_shuff = df_shuff.groupBy("cname").sum("revenue")
sum_df.display()

With AQE

In [0]:
spark.conf.set("spark.sql.shuffle.partitions","200")
#still important to set bcos AQE cannot exceed 200 partitions at any cost.
# AQE will automatic decide max partitions required. Will never exceed shuffle partitions limit.

In [0]:
df_aqe = spark.createDataFrame(data_list, schema)
sum_df_aqe = df_aqe.groupBy("cname").sum("revenue")
sum_df.display()

In [0]:
spark.conf.get("spark.sql.shuffle.partitions")
spark.conf.get("spark.sql.adaptive.coalescePartitions.minPartitionNum")
spark.conf.get("spark.sql.adaptive.advisoryPartitionSizeInBytes")
spark.conf.get("spark.sql.adaptive.skewJoin.enabled")
spark.conf.get("spark.sql.adaptive.skewJoin.skewedPartitionFactor")